# Глава 10. Градиентный спуск и бустинг — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Градиентный спуск: шаг за шагом к минимуму

В главе 9 веса линейной регрессии нашлись одной формулой. Для большинства моделей такой формулы нет — веса ищут постепенно, двигаясь «под горку» по функции ошибки. Это градиентный спуск: на нём обучаются бустинг и нейросети. Когда формула всё же дешевле спуска — в следующем уроке.

Теория — на сайте курса: `ch10/01-gradient.html`. Задачи «собери код из строк» (1) остались на сайте.

### Потеря в точке · разминка

Напишите функцию `loss(w)` — MSE модели `w * x` на данных `x`, `y` — и сохраните `l5 = loss(5)` и `l10 = loss(10)`. В `better` сохраните то из чисел 5 и 10, при котором ошибка меньше.

*Подсказки (2): `dsc.hint('loss-at')`*

In [ ]:
%%task ch10/01-gradient/loss-at
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.5, 6.0])
y = np.array([18.0, 25.0, 34.0, 44.0, 55.0])

### Численная производная · разминка

Напишите функцию `num_grad(f, w, eps=1e-5)`, которая оценивает производную функции `f` в точке `w` по формуле (f(w + eps) − f(w − eps)) / (2·eps). Проверьте её на f(w) = w²: сохраните `g3 = num_grad(lambda w: w ** 2, 3)` (должно быть около 6).

Затем сверьте формулу производной из урока с численной оценкой на пяти доставках (`loss` и `grad` в заготовке): сохраните `g5_formula = grad(5)` и `g5_numeric = num_grad(loss, 5)`.

*Подсказки (2): `dsc.hint('numeric-grad')`*

In [ ]:
%%task ch10/01-gradient/numeric-grad
import numpy as np
km = np.array([1.0, 2.0, 3.0, 4.5, 6.0])
minutes = np.array([18.0, 25.0, 34.0, 44.0, 55.0])

def loss(w):
    return np.mean((w * km - minutes) ** 2)

def grad(w):
    return np.mean(2 * (w * km - minutes) * km)

### Три шага обучения · база

Запустите спуск из `w = 0` по 30 шагов с тремя шагами обучения: 0.001, 0.01 и 0.1 (функции `loss` и `grad` в заготовке). Сохраните словарь `final` — `{lr: значение MSE после 30 шагов}` и `diverged` — список шагов обучения, при которых итоговая ошибка больше, чем в начале (при `w = 0`).

*Подсказки (2): `dsc.hint('lr-modes')`*

In [ ]:
%%task ch10/01-gradient/lr-modes
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.5, 6.0])
y = np.array([18.0, 25.0, 34.0, 44.0, 55.0])

def loss(w):
    return np.mean((w * x - y) ** 2)

def grad(w):
    return np.mean(2 * (w * x - y) * x)

### Кривая обучения · база

Запустите спуск с `lr = 0.01` из `w = 0` на 40 шагов и сохраните список `history` — MSE после каждого шага (40 чисел). Нарисуйте его линией, подпишите ось X `"шаг"`, ось Y `"MSE"`. В `steps_to_30` сохраните номер первого шага (считая с 1), после которого MSE стала меньше 30.

*Подсказки (2): `dsc.hint('history')`*

In [ ]:
%%task ch10/01-gradient/history
import numpy as np
import matplotlib.pyplot as plt
x = np.array([1.0, 2.0, 3.0, 4.5, 6.0])
y = np.array([18.0, 25.0, 34.0, 44.0, 55.0])

def loss(w):
    return np.mean((w * x - y) ** 2)

def grad(w):
    return np.mean(2 * (w * x - y) * x)

### Правило остановки · база

Напишите `descend(lr, tol=1e-6, max_steps=10000)` как в разборе урока: спуск из `w = 0`, остановка, когда потеря за шаг изменилась меньше чем на `tol`; вернуть `(w, число шагов)`. Сохраните `steps` — словарь `{lr: число шагов}` для `lr` из `[0.001, 0.01, 0.03]` и `w_found` — словарь `{lr: w}` (4 знака). Затем выясните, что будет при `lr = 0.1`: сохраните `steps_01` и `w_01` (как есть, без округления). При `lr = 0.1` numpy напечатает предупреждения `RuntimeWarning: overflow` — это ожидаемо: числа перестают помещаться в float. Чтобы их не видеть, вызов можно обернуть в `with np.errstate(all="ignore"):`.

*Подсказки (3): `dsc.hint('stop-rule')`*

In [ ]:
%%task ch10/01-gradient/stop-rule
def descend(lr, tol=1e-6, max_steps=10000):
    w, prev = 0.0, loss(0.0)
    ...

### Вес и свободный член · посложнее

Найдите спуском веса `w` и `b` модели `w * x + b` для данных из заготовки: старт 0, `lr = 0.02`, 3000 шагов. Сохраните `w`, `b` (2 знака) и `exact` — решение `np.polyfit(x, y, 1)` (тоже 2 знака). В `close_enough` — `True`, если спуск отличается от точного решения меньше чем на 0.05 по обоим параметрам.

*Подсказки (2): `dsc.hint('two-params')`*

In [ ]:
%%task ch10/01-gradient/two-params
import numpy as np
x = np.array([0.5, 1.5, 2.0, 3.5, 5.0, 7.0])
y = np.array([14.0, 19.0, 22.0, 31.0, 38.0, 52.0])

### Спуск по другой потере · со звёздочкой

Обучите ту же модель `w * x + b` спуском по **средней абсолютной ошибке** (MAE). Её «производная» по w — mean(sign(ошибка) · x), по b — mean(sign(ошибка)), где `np.sign` даёт −1, 0 или 1. Данные — из заготовки, в них есть выброс. Старт 0, `lr = 0.05`, 5000 шагов. Сохраните `w_mae`, `b_mae` (2 знака) и для сравнения `w_mse` — наклон `np.polyfit(x, y, 1)[0]` (2 знака). Какой наклон ближе к 7 (настоящему наклону без выброса)? Сохраните `robust` — `"MAE"` или `"MSE"`.

*Подсказки (2): `dsc.hint('mae-gd')` · задача по желанию*

In [ ]:
%%task ch10/01-gradient/mae-gd
import numpy as np
x = np.array([0.5, 1.5, 2.0, 3.5, 5.0, 7.0, 6.0])
y = np.array([14.0, 20.0, 24.0, 34.0, 45.0, 59.0, 120.0])   # последний заказ — сбой часов

## Урок 2. Численные методы: цена формулы и деление пополам

Для линейной регрессии есть точная формула, а спуск идёт к тому же ответу тысячами шагов. Разбираемся, когда формула дешевле спуска, а когда она проигрывает по времени и памяти, и знакомимся ещё с одним численным методом — делением пополам. С его помощью найдём, какое время доставки обещать гостям.

Теория — на сайте курса: `ch10/01b-numeric.html`.

### Операции на салфетке · разминка

В таблице 1000 строк и 10 столбцов (вместе со столбцом единиц), спуску нужно 100 шагов. Посчитайте по формулам урока, сколько умножений потребуется:

- `normal_ops` — нормальному уравнению ($n \cdot d^2 + d^3$);
- `gd_ops` — спуску ($2 \cdot n \cdot d \cdot k$);
- `cheaper` — `"формула"` или `"спуск"`, смотря что дешевле;
- `steps_equiv` — сколько целых шагов спуска стоят столько же, сколько одна формула: `normal_ops // (2 * n * d)`.

*Подсказки (2): `dsc.hint('ops-hand')`*

In [ ]:
%%task ch10/01b-numeric/ops-hand
n, d, k = 1000, 10, 100

### Формула или спуск · база

Напишите две функции подсчёта умножений: `ops_normal(n, d)` — нормальное уравнение ($n \cdot d^2 + d^3$) и `ops_gd(n, d, steps)` — спуск ($2 \cdot n \cdot d \cdot steps$). Для трёх таблиц из заготовки и 500 шагов соберите DataFrame `cost`: индекс — название таблицы, столбцы `normal`, `gd` и `cheaper` (`"формула"` или `"спуск"`). Затем сохраните:

- `ratio_text` — во сколько раз спуск дешевле формулы для текстов отзывов (`int`);
- `mem_gb` — сколько гигабайт займёт $X^\top X$ для текстов, если одно число — 8 байт (1 ГБ = 10⁹ байт);
- `d_break` — при $n = 100\,000$ и 500 шагах наименьшее $d$ от 1 до 5000, при котором формула становится дороже спуска.

*Подсказки (3): `dsc.hint('ops-count')`*

In [ ]:
%%task ch10/01b-numeric/ops-count
import pandas as pd

tables = {
    "доставки": (1841, 4),
    "чеки за год": (1_000_000, 20),
    "тексты отзывов": (100_000, 50_000),
}

### Сколько делений нужно · разминка

Деление пополам каждый шаг вдвое укорачивает отрезок. Не запуская сам поиск, посчитайте:

- `len_after_5` — длину отрезка `[0, 100]` после 5 делений;
- `steps_100` — сколько делений нужно, чтобы отрезок `[0, 100]` стал не длиннее 0.01;
- `steps_300` — то же для отрезка `[0, 300]` и допуска 0.01 (как в примере с обещанием времени).

Число шагов можно найти циклом «пока длина больше допуска — делим» или формулой $\lceil \log_2(\text{длина} / \text{допуск}) \rceil$ — в Python `math.ceil(math.log2(...))`.

*Подсказки (2): `dsc.hint('bisect-steps')`*

In [ ]:
%%task ch10/01b-numeric/bisect-steps
import math

### Деление пополам · база

Напишите функцию `bisect(f, lo, hi, tol=1e-6)`: она ищет ноль функции `f` на отрезке `[lo, hi]`, где у `f(lo)` и `f(hi)` разные знаки, и возвращает пару `(x, steps)`: середину последнего отрезка и число сделанных делений. С её помощью найдите:

- `root3` — кубический корень из 2: ноль `x**3 - 2` на `[0, 2]` (6 знаков) и `steps3` — сколько шагов понадобилось;
- `w_min` — дно ямы для данных из заготовки: ноль производной `grad(w)` на `[0, 30]` (3 знака). Сравните с точным `exact = (x * y).sum() / (x ** 2).sum()` (тоже 3 знака).

*Подсказки (3): `dsc.hint('bisect')`*

In [ ]:
%%task ch10/01b-numeric/bisect
import numpy as np
x = np.array([0.8, 1.5, 2.5, 3.0, 5.0])
y = np.array([10.0, 16.0, 24.0, 31.0, 47.0])

def grad(w):
    return np.mean(2 * (w * x - y) * x)

def bisect(f, lo, hi, tol=1e-6):
    ...

### Цена обещания · база

Директор спрашивает: насколько дольше придётся обещать, если опаздывать не в 10 %, а в 5 % доставок? А если смириться с 20 %? Для каждого уровня опозданий из `levels` найдите делением пополам обещание в минутах: ноль функции «доля доставок дольше $t$ − уровень» на `[0, 300]` с `tol=0.01`, результат округлите через `round`. Функция `bisect` из урока уже есть. Сохраните:

- `promise` — словарь `{уровень: обещание}` (`int`);
- `late` — словарь `{уровень: доля опозданий при этом обещании}` (3 знака);
- `extra_for_5` — на сколько минут обещание при 5 % дольше, чем при 10 %.

*Подсказки (2): `dsc.hint('promise-levels')`*

In [ ]:
%%task ch10/01b-numeric/promise-levels
# уже есть: функция bisect(f, lo, hi, tol) из урока
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
actual = d.loc[d["actual_min"].between(1, 299), "actual_min"].values
levels = [0.05, 0.10, 0.20]

### Обещание для каждого типа курьера · посложнее

Сейчас «Зерно» обещает одно и то же время всем курьерам, хотя пешие медленнее. Для каждого типа курьера (`авто`, `вело`, `пеший`) найдите делением пополам наименьшее целое обещание (в минутах), при котором опаздывает не больше 10 % доставок этого типа. Используйте функцию `bisect` из заготовки на отрезке `[0, 300]` с `tol=0.01` и округлите результат через `round`. Сохраните словари с ключами-типами:

- `promise` — найденное обещание (`int`);
- `late_at_promise` — доля опозданий при этом обещании (3 знака);
- `q90` — 90-й процентиль времени `np.quantile(..., 0.9)` для сравнения.

В `slowest` сохраните тип курьера с самым долгим обещанием.

*Подсказки (3): `dsc.hint('promise-time')`*

In [ ]:
%%task ch10/01b-numeric/promise-time
# уже есть: функция bisect(f, lo, hi, tol) из урока
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

## Урок 3. Спуск для многих признаков

Переносим градиентный спуск на настоящую таблицу: градиент для всех весов сразу одной матричной формулой, масштабирование, без которого спуск буксует или взрывается, и стохастический спуск по маленьким порциям данных — так учатся модели на миллионах строк. Как сделать сам шаг умнее — в следующем уроке.

Теория — на сайте курса: `ch10/02-gd-many.html`. Задачи «собери код из строк» (1) остались на сайте.

### Градиент матрицей · разминка

Для маленькой таблицы `X1` (уже со столбцом единиц) и весов `w` посчитайте градиент MSE формулой `2 / n * X1.T @ (X1 @ w - y)` и сохраните его в `g`. В `direction` сохраните список: для каждого веса `"вверх"`, если вес надо увеличивать (градиент отрицательный), иначе `"вниз"`.

*Подсказки (2): `dsc.hint('grad-vec')`*

In [ ]:
%%task ch10/02-gd-many/grad-vec
import numpy as np
X1 = np.array([[1, 1.0], [1, 2.0], [1, 3.0]])
y = np.array([5.0, 7.0, 9.0])
w = np.array([0.0, 4.0])

### Один шаг руками · разминка

Три доставки, один признак и столбец единиц: `X1 = [[1, 1], [1, 2], [1, 3]]`, ответы `y = [3, 5, 7]`, стартовые веса `w = [0, 0]`. Посчитайте градиент MSE `g = 2 / n * X1.T @ (X1 @ w - y)` и веса после одного шага `w_new = w - lr * g` при `lr = 0.1`. Сохраните также `mse_before` и `mse_after` — MSE до и после шага.

*Подсказки (2): `dsc.hint('ch10/02-gd-many/one-step')`*

In [ ]:
%%task ch10/02-gd-many/one-step
import numpy as np
X1 = np.array([[1, 1], [1, 2], [1, 3]], dtype=float)
y = np.array([3, 5, 7], dtype=float)
w = np.zeros(2)
lr = 0.1

### Спуск на доставках · база

Напишите функцию `gd(X1, y, lr, steps)`, возвращающую веса после `steps` шагов спуска из нулей. Обучите линейную модель на масштабированных признаках `distance_km`, `items`, `walk` (`Xs` уже в заготовке) со столбцом единиц: `lr = 0.1`, 200 шагов. Сохраните веса `w` и `mse` — MSE на тех же данных (2 знака).

*Подсказки (2): `dsc.hint('gd-fit')`*

In [ ]:
%%task ch10/02-gd-many/gd-fit
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk"]].values
y = ok["actual_min"].values
Xs = (X - X.mean(axis=0)) / X.std(axis=0)

### Самый большой безопасный шаг · база

Для масштабированных признаков (как в прошлой задаче, со столбцом единиц) найдите среди шагов `[0.1, 0.3, 0.5, 0.7, 0.9, 1.1]` самый большой, при котором после 100 шагов MSE меньше 61. Сохраните словарь `res` `{шаг: MSE после 100 шагов}` (для взорвавшихся значений будет `inf` или `nan` — это нормально) и `best_lr`.

*Подсказки (2): `dsc.hint('max-lr')`*

In [ ]:
%%task ch10/02-gd-many/max-lr
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk"]].values
y = ok["actual_min"].values
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
X1 = np.column_stack([np.ones(len(Xs)), Xs])

### Мини-батчи · посложнее

Обучите модель стохастическим спуском: батчи по 64 строки, шаг 0.05, 3 эпохи, генератор `np.random.default_rng(1)`, порядок строк перемешивается в начале каждой эпохи (`rng.permutation`). Сохраните список `epoch_mse` — MSE на всех данных после каждой эпохи (3 числа) и `steps` — сколько всего шагов (обновлений весов) было сделано.

*Подсказки (2): `dsc.hint('ch10/02-gd-many/minibatch')`*

In [ ]:
%%task ch10/02-gd-many/minibatch
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk"]].values
y = ok["actual_min"].values
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
X1 = np.column_stack([np.ones(len(Xs)), Xs])

### Сколько шагов до цели · база

Посчитайте, за сколько шагов полного (не стохастического) спуска MSE на доставках с четырьмя признаками (`distance_km`, `items`, `walk`, `amount`; таблицы `X`, `Xs` и ответ `y` уже есть в заготовке) опустится ниже 61. Напишите `steps_to(A, lr, goal=61, max_steps=20000)` — номер первого шага, после которого MSE < goal, или `None`, если не дошли. Сохраните `scaled` — словарь `{lr: шаги}` для масштабированных признаков и `lr` из `[0.1, 0.5]`, и `raw` — то же для исходных признаков и `lr` из `[1e-6, 2e-6]`.

*Подсказки (2): `dsc.hint('to-goal')`*

In [ ]:
%%task ch10/02-gd-many/to-goal
# уже есть: X — четыре признака (distance_km, items, walk, amount) без масштаба,
# Xs — они же масштабированные, y — время доставки
import numpy as np

def steps_to(A, lr, goal=61, max_steps=20000):
    X1 = np.column_stack([np.ones(len(A)), A])
    w = np.zeros(X1.shape[1])
    ...

### Уменьшающийся шаг · посложнее

SGD с постоянным шагом «дрожит» вокруг минимума. Сравните постоянный шаг 0.05 и уменьшающийся `lr_t = 0.05 / (1 + t / 200)`, где t — номер шага с начала обучения (с 0). Признаки — масштабированные `distance_km`, `items`, `walk` (заготовка: `X1` уже со столбцом единиц), батч 32, 20 эпох, порядок строк — `rng = np.random.default_rng(0)` (свой генератор для каждого варианта). После каждой эпохи запишите MSE на всех данных. Сохраните `last5_const`, `last5_decay` — MSE последних пяти эпох (списки, 3 знака) и `jitter` — словарь `{"const": …, "decay": …}` с размахом (max − min) этих пяти значений (3 знака).

*Подсказки (3): `dsc.hint('ch10/02-gd-many/decay')`*

In [ ]:
%%task ch10/02-gd-many/decay
import numpy as np

def sgd(decay, epochs=20, lr0=0.05, size=32):
    rng = np.random.default_rng(0)
    w = np.zeros(X1.shape[1])
    t = 0
    history = []
    ...
    return history

### SGDRegressor и порядок данных · со звёздочкой

Сравните `make_pipeline(StandardScaler(), SGDRegressor(random_state=r))` для `r` в `[0, 1, 2, 3, 4]` на доставках (признаки `distance_km`, `items`, `walk`; цель `actual_min`). Сохраните список `maes` — MAE на тех же данных для каждого `r` (3 знака), `spread` — разницу между наибольшей и наименьшей (3 знака) и `mae_exact` — MAE `LinearRegression` (3 знака).

*Подсказки (2): `dsc.hint('sgd-sklearn')` · задача по желанию*

In [ ]:
%%task ch10/02-gd-many/sgd-sklearn
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk"]]
y = ok["actual_min"]

## Урок 4. Оптимизаторы: инерция, Adam и затухание весов

Обычный шаг спуска ползёт по пологим склонам и боится признаков разного масштаба. Нейросети учатся не голым спуском, а оптимизаторами — правилами шага, которые чинят обе беды. Разбираем инерцию и Adam руками и на доставках, а заодно смотрим, как L2-штраф из Ridge выглядит внутри шага.

Теория — на сайте курса: `ch10/02b-optimizers.html`.

### Три шага с инерцией · разминка

Склон с постоянным градиентом `g = -1`, шаг `lr = 0.2`, `beta = 0.5`, старт `v = 0`, `w = 0`. Сделайте три шага с инерцией по правилу урока (v ← β·v + g, w ← w − lr·v) и сохраните:

- `ws` — список весов после каждого шага (3 числа, 4 знака);
- `plain_step` — длину шага обычного спуска без инерции: `lr * abs(g)`;
- `limit_step` — к какой длине стремится шаг с инерцией: `lr * abs(g) / (1 - beta)`.

*Подсказки (2): `dsc.hint('momentum-hand')`*

In [ ]:
%%task ch10/02b-optimizers/momentum-hand
g, lr, beta = -1.0, 0.2, 0.5
v, w = 0.0, 0.0

### Спуск с инерцией · посложнее

Проверим инерцию на модели «минуты = w·км + b» с пяти доставок урока «Градиентный спуск: шаг за шагом к минимуму» — той самой, где свободный член «тянулся» медленно. Правило шага — из раздела выше: v ← β·v + градиент, w ← w − lr·v, отдельно для `w` и для `b`. Запустите 3000 шагов с `lr = 0.02` и β из `[0, 0.5, 0.9]` (β = 0 — обычный спуск). Для каждого β найдите номер первого шага, после которого MSE отличается от минимально возможной `best_mse` (она уже посчитана в заготовке по точному решению `np.polyfit`) меньше чем на 0.01: `steps_by_beta` — словарь `{β: шаг}`. В `final_wb` — `(w, b)` при β = 0.9 после 3000 шагов (3 знака).

*Подсказки (2): `dsc.hint('momentum')`*

In [ ]:
%%task ch10/02b-optimizers/momentum
# уже есть: km, minutes (пять доставок) и best_mse — MSE точного решения np.polyfit
import numpy as np

def run(beta, lr=0.02, steps=3000):
    w, b, vw, vb = 0.0, 0.0, 0.0, 0.0
    history = []
    ...
    return w, b, history

### Шаг Adam руками · база

Напишите функцию `adam_step(w, g, m, s, t, lr=0.1, b1=0.9, b2=0.999, eps=1e-8)`: она делает один шаг Adam по формулам урока и возвращает тройку `(w, m, s)` — новые веса, сглаженный градиент и сглаженный квадрат градиента. `t` — номер шага, начиная с 1.

На трёх доставках из разбора первого шага Adam (заготовка: `X1`, `y`, старт `w = (0, 5)`) сделайте 3 шага Adam с `lr = 0.1`. Градиент на каждом шаге — `2 / n * X1.T @ (X1 @ w - y)`. Сохраните:

- `history` — список весов после каждого шага (3 массива по 2 числа, округлите до 4 знаков);
- `first_step` — сдвиг весов на первом шаге, `w` после шага минус `w` до (4 знака);
- `raw_first` — каким был бы первый сдвиг **без** поправки: `-lr * m / (np.sqrt(s) + eps)` с неисправленными `m` и `s` после первого шага (4 знака).

*Подсказки (3): `dsc.hint('adam-step')`*

In [ ]:
%%task ch10/02b-optimizers/adam-step
import numpy as np
X1 = np.array([[1, 1], [1, 2], [1, 4.0]])    # столбец единиц и км
y = np.array([18, 25, 40.0])
n = len(y)

def adam_step(w, g, m, s, t, lr=0.1, b1=0.9, b2=0.999, eps=1e-8):
    ...

### Шаг Adam тоже подбирают · база

Adam прощает немасштабированные признаки, но не любой шаг. На доставках без масштаба (заготовка: `X1` со столбцом единиц и суммой заказа в рублях, `y`) запустите Adam по 2000 шагов из нулевых весов с каждым `lr` из `[0.005, 0.01, 0.05, 0.2, 1.0]`. Функция одного шага `adam_step` уже есть — та же, что в задаче «Шаг Adam руками». Сохраните:

- `mse_by_lr` — словарь `{lr: MSE после 2000 шагов}` (1 знак);
- `good` — список `lr` (по возрастанию), при которых MSE меньше 61 (минимум на этих данных — 60.0).

*Подсказки (2): `dsc.hint('adam-lr')`*

In [ ]:
%%task ch10/02b-optimizers/adam-lr
# уже есть: adam_step(w, g, m, s, t, lr) из задачи «Шаг Adam руками»
import numpy as np
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk", "amount"]].values.astype(float)   # без масштаба
y = ok["actual_min"].values
X1 = np.column_stack([np.ones(len(X)), X])
n = len(y)

### Затухание веса · разминка

Затухание весов из раздела про L2-штраф: на каждом шаге вес умножается на $1 - 2\eta\lambda$. Пусть градиент по весу равен нулю (признак ничего не объясняет), $w = 10$, $\eta = 0.1$, $\lambda = 0.5$. Сохраните:

- `factor` — множитель $1 - 2\eta\lambda$;
- `w5` — вес после 5 таких шагов (4 знака);
- `w5_small` — вес после 5 шагов при $\lambda = 0.05$ (4 знака).

*Подсказки (2): `dsc.hint('ch10/02b-optimizers/decay')`*

In [ ]:
%%task ch10/02b-optimizers/decay
w0, lr, lam = 10.0, 0.1, 0.5

### L2-штраф в спуске и Ridge · посложнее

Напишите функцию `gd_l2(X1, y, lam, lr=0.1, steps=1000)`: спуск по MSE со штрафом $\lambda \sum w_j^2$, где свободный член `w[0]` не штрафуется. Признаки в заготовке уже стандартизированы, первый столбец `X1` — единицы. Для `lam` из `[0, 0.1, 1.0]` сохраните словари с ключами-λ:

- `weights` — веса после спуска (массивы, свободный член первым);
- `norms` — длина вектора весов **без** свободного члена, `np.linalg.norm(w[1:])` (3 знака);
- `mse_train` — MSE на этих данных (2 знака).

Ещё сохраните `alpha_ridge` — какой `alpha` у `Ridge` соответствует `lam = 1.0` на этих данных, и `same_as_ridge` — совпадают ли веса спуска при `lam = 1.0` с `Ridge(alpha=alpha_ridge)` (`np.allclose(..., atol=1e-6)`).

*Подсказки (3): `dsc.hint('l2-grad')`*

In [ ]:
%%task ch10/02b-optimizers/l2-grad
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["walk"] = (ok["courier"] == "пеший").astype(int)
X = ok[["distance_km", "items", "walk"]].values.astype(float)
y = ok["actual_min"].values
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
X1 = np.column_stack([np.ones(len(Xs)), Xs])

def gd_l2(X1, y, lam, lr=0.1, steps=1000):
    ...

## Урок 5. Бустинг: каждое дерево исправляет ошибки предыдущих

Случайный лес строит много деревьев независимо и усредняет. Бустинг устроен иначе: деревья строятся по очереди, и каждое следующее учится на ошибках всех предыдущих. Собираем бустинг руками из маленьких деревьев, а потом сверяем с sklearn.

Теория — на сайте курса: `ch10/03-boosting.html`. Задачи «собери код из строк» (1) остались на сайте.

### Бустинг на бумаге · разминка

Повторите пример с четырьмя доставками кодом, но с шагом η = 0.3 и тремя шагами. «Пень» здесь — просто средний остаток в каждой группе по признаку `walk`. Сохраните `history` — список прогнозов после каждого шага (каждый прогноз — список из 4 чисел, 2 знака), начиная со старта (всего 4 списка), и `gap_left` — сколько минут ещё не хватает группе пеших до их среднего (40) после трёх шагов (2 знака).

*Подсказки (2): `dsc.hint('toy-boost')`*

In [ ]:
%%task ch10/03-boosting/toy-boost
import numpy as np

lr = 0.3
f = np.full(4, y.mean())
history = [f.round(2).tolist()]

### Остатки · разминка

Прогноз `f` — пока просто среднее `y`. Сохраните остатки `r = y - f` и `mae0` — среднюю абсолютную ошибку такого прогноза.

*Подсказки (2): `dsc.hint('ch10/03-boosting/residuals')`*

In [ ]:
%%task ch10/03-boosting/residuals
import numpy as np
y = np.array([22.0, 35.0, 41.0, 28.0, 54.0])

### Один шаг бустинга · разминка

Сделайте один шаг бустинга руками: «дерево» уже обучено и предсказывает поправки `tree_pred`. Обновите прогноз `f` с шагом 0.1 и сохраните новый прогноз `f1` и его MAE `mae1`.

*Подсказки (2): `dsc.hint('ch10/03-boosting/one-step')`*

In [ ]:
%%task ch10/03-boosting/one-step
import numpy as np
y = np.array([22.0, 35.0, 41.0, 28.0, 54.0])
f = np.full(5, y.mean())
tree_pred = np.array([-12.0, -2.0, 4.0, -8.0, 16.0])

### Бустинг руками · база

Напишите функцию `boost(X_tr, y_tr, X_te, n, lr, depth)`, которая делает `n` шагов бустинга с деревьями `DecisionTreeRegressor(max_depth=depth, random_state=0)` и возвращает прогноз для `X_te`. Посчитайте MAE на тесте для `(n=30, lr=0.1, depth=3)` и `(n=30, lr=0.3, depth=3)` и сохраните `mae_01`, `mae_03` (2 знака).

*Подсказки (2): `dsc.hint('manual-boost')`*

In [ ]:
%%task ch10/03-boosting/manual-boost
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Сверка с sklearn · база

Сравните ваш `boost` (50 деревьев, шаг 0.1, глубина 3) с `GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=0)`. Сохраните `mae_mine`, `mae_sk` (2 знака) и `corr` — корреляцию прогнозов двух моделей на тесте (4 знака).

*Подсказки (2): `dsc.hint('vs-sklearn')`*

In [ ]:
%%task ch10/03-boosting/vs-sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

def boost(X_tr, y_tr, X_te, n, lr, depth):
    f_tr = np.full(len(y_tr), y_tr.mean())
    f_te = np.full(len(X_te), y_tr.mean())
    for _ in range(n):
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_tr, y_tr - f_tr)
        f_tr = f_tr + lr * tree.predict(X_tr)
        f_te = f_te + lr * tree.predict(X_te)
    return f_te

### Пни против деревьев · посложнее

Сравните бустинг из «пней» (деревьев глубины 1) и из деревьев глубины 4: `GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=d, random_state=0)` для `d` = 1 и 4. Сохраните `mae_stumps`, `mae_deep` (2 знака) и `winner` — `"пни"` или `"глубина 4"`.

*Подсказки (2): `dsc.hint('depth-one')`*

In [ ]:
%%task ch10/03-boosting/depth-one
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Шаг и число деревьев · база

Проверьте правило «уменьшили шаг вдвое — деревьев нужно вдвое больше». Обучите `GradientBoostingRegressor(max_depth=3, random_state=0)` на доставках (данные и разбиение — в заготовке) для пар `(learning_rate, n_estimators)`: `(0.3, 100)`, `(0.1, 300)`, `(0.05, 600)` и `(0.02, 300)`. Сохраните `mae` — словарь `{"0.3x100": …, "0.1x300": …, "0.05x600": …, "0.02x300": …}` (MAE на тесте, 3 знака) и `undertrained` — ключ пары, которая заметно хуже остальных (больше чем на 0.1 минуты от лучшей).

*Подсказки (2): `dsc.hint('lr-trees')`*

In [ ]:
%%task ch10/03-boosting/lr-trees
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

pairs = [(0.3, 100), (0.1, 300), (0.05, 600), (0.02, 300)]
mae = {}

### Бустинг по MAE · со звёздочкой

Сделайте ручной бустинг по **MAE**: «остаток» для MAE — знак ошибки `np.sign(y - f)`, а не сама ошибка. Старт — **медиана** `y_train`, 100 деревьев глубины 3 (`random_state=0`), шаг 1.0 — прогноз дерева (средний знак в листе, от −1 до 1) прибавляется целиком. Сохраните `mae_l1` — MAE на тесте (2 знака) и `mae_l2` — MAE бустинга по MSE из `GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)` (2 знака).

*Подсказки (2): `dsc.hint('mae-boost')` · задача по желанию*

In [ ]:
%%task ch10/03-boosting/mae-boost
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## Урок 6. Настройка бустинга

У бустинга, в отличие от леса, слишком много деревьев вредит: в какой-то момент он начинает учить шум. Учимся видеть этот момент на кривой обучения, останавливаться вовремя, подбирать шаг и глубину и знакомимся с быстрым HistGradientBoosting, который сам справляется с пропусками и категориями.

Теория — на сайте курса: `ch10/04-tuning.html`. Задачи «собери код из строк» (1) остались на сайте.

### Кривая по деревьям · разминка

Обучите `GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)` и с помощью `staged_predict` соберите список `curve` — MAE на тесте после каждого дерева (100 чисел). Сохраните `best_n` — число деревьев с наименьшей ошибкой.

*Подсказки (2): `dsc.hint('staged')`*

In [ ]:
%%task ch10/04-tuning/staged
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Шаг обучения руками · разминка

Упрощённый бустинг на одном числе: ответ `y = 10`, стартовый прогноз 0, и каждое новое дерево **точно** угадывает текущий остаток, но добавляется с весом `lr = 0.1`. Тогда после каждого дерева прогноз растёт на 10 % оставшегося остатка. Посчитайте циклом список `preds` — прогнозы после 1, 2, …, 40 деревьев — и `n_needed` — первое число деревьев, при котором прогноз не меньше 9.5.

*Подсказки (2): `dsc.hint('shrink')`*

In [ ]:
%%task ch10/04-tuning/shrink
y, lr = 10, 0.1
pred = 0
preds = []

### Где начинается переобучение · база

Обучите `GradientBoostingRegressor(n_estimators=400, learning_rate=0.3, max_depth=3, random_state=0)`. По `staged_predict` соберите MAE на обучении (`tr`) и на тесте (`te`) — по 400 чисел. Сохраните `best_n` (минимум на тесте), `te_best`, `te_last` (MAE на тесте при лучшем числе деревьев и при 400, 3 знака) и `tr_last` (MAE на обучении при 400, 3 знака).

*Подсказки (2): `dsc.hint('train-vs-test')`*

In [ ]:
%%task ch10/04-tuning/train-vs-test
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Остановка по валидации · база

Найдите число деревьев честно: отделите от обучающих данных валидацию (`train_test_split(X_train, y_train, test_size=0.2, random_state=0)`), обучите бустинг на 600 деревьев (шаг 0.1, глубина 3, `random_state=0`) на оставшейся части, найдите по `staged_predict` лучшее число деревьев на валидации `best_n`, затем обучите модель с `n_estimators=best_n` на **всех** обучающих данных и сохраните `test_mae` (2 знака).

*Подсказки (2): `dsc.hint('early-stop')`*

In [ ]:
%%task ch10/04-tuning/early-stop
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Помогает ли подвыборка · база

Параметр `subsample` строит каждое дерево на случайной доле строк — как бутстрэп в лесу. Проверьте на доставках: `GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0)` с `subsample` из `[1.0, 0.7, 0.5]`. Сохраните `mae_by_sub` — словарь `{subsample: MAE на тесте}` (3 знака) и `helps` — `True`, если хоть одно значение меньше 1.0 улучшило MAE больше чем на 0.01.

*Подсказки (2): `dsc.hint('subsample')`*

In [ ]:
%%task ch10/04-tuning/subsample
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

### Категории без флагов · посложнее

Повторите разбор урока: `HistGradientBoostingRegressor(max_iter=300, max_depth=3, categorical_features="from_dtype", random_state=0)` на доставках в двух вариантах — признаки через `pd.get_dummies` и признаки с курьером, погодой и кофейней как `category`. Разбиение одинаковое: `train_test_split(..., test_size=0.25, random_state=42)`. Сохраните `res` — словарь `{"dummies": (число столбцов, MAE), "category": (число столбцов, MAE)}` (MAE — 3 знака) и `n_categories` — общее число разных значений в трёх категориальных столбцах.

*Подсказки (2): `dsc.hint('native-cat')`*

In [ ]:
%%task ch10/04-tuning/native-cat
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

res = {}

### Бустинг без заполнения пропусков · посложнее

На `guests_churn.csv` (признаки — `get_dummies` без `fillna`, `card_blocked` исключить) обучите `HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_depth=3, random_state=0)` и `RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0)` (лесу пропуски заполните нулём). Сохраните `auc_hgb`, `auc_rf` (3 знака) и `nan_in_train` — число пропусков в `X_train`.

*Подсказки (2): `dsc.hint('hgb-nan')`*

In [ ]:
%%task ch10/04-tuning/hgb-nan
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Подбор глубины и шага · со звёздочкой

Подберите для `HistGradientBoostingClassifier(max_iter=200, random_state=0)` на оттоке параметры по сетке `{"learning_rate": [0.05, 0.1], "max_depth": [2, 3, 5]}` через `GridSearchCV(..., cv=3, scoring="roc_auc")` на обучающей части. Сохраните `best_params` (словарь), `cv_auc` — лучший средний AUC на кросс-валидации и `test_auc` — AUC лучшей модели на тесте (оба 3 знака).

*Подсказки (2): `dsc.hint('grid-boost')` · задача по желанию*

In [ ]:
%%task ch10/04-tuning/grid-boost
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 7. Проект: прогноз дневной выручки

Финал главы. Управляющие кофейнями планируют смены и закупки по ощущениям. Строим прогноз выручки каждой кофейни на каждый день: честная проверка «по времени», простые ориентиры, линейная модель с ручными взаимодействиями, бустинг с подбором по валидации — и ловушка, которую деревья не видят: рост бизнеса.

Теория — на сайте курса: `ch10/05-project.html`.

### Шаг 1. Формат и выходные · база

Посчитайте среднюю выручку по формату и выходным (суббота и воскресенье) — таблицу `pivot` (строки — формат, столбцы — 0/1 «выходной», 1 знак). Сохраните `weekend_ratio` — для каждого формата отношение выручки в выходной к будней (Series, 2 знака), и `most_weekday` — формат, который сильнее всего зависит от будней (наименьшее отношение).

*Подсказки (2): `dsc.hint('ch10/05-project/look')`*

In [ ]:
%%task ch10/05-project/look
import pandas as pd
s = pd.read_csv("daily_sales.csv", parse_dates=["date"])
s["weekend"] = (s["date"].dt.weekday >= 5).astype(int)

### Шаг 2. Признаки и разбиение по времени · база

Напишите функцию `prep(s)`, которая добавляет `weekday`, `month`, `weekend`, `day_n` (дней от первой даты в данных) и возвращает таблицу признаков: `temp`, `rain`, `holiday`, `promo`, `weekday`, `month`, `weekend`, `day_n` плюс флаги `shop_id` и `format` (`pd.get_dummies(..., columns=["shop_id", "format"], dtype=int)`). Разделите данные по времени: `train` — даты до 2025-07-01, `test` — с этой даты. Сохраните `X`, маску `tr` (True для обучающих строк), `n_train`, `n_test`.

*Подсказки (2): `dsc.hint('ch10/05-project/split')`*

In [ ]:
%%task ch10/05-project/split
import pandas as pd
s = pd.read_csv("daily_sales.csv", parse_dates=["date"])

### Шаг 3. Ориентиры · база

Посчитайте на тесте MAE двух ориентиров:

- `mae_lag7` — прогноз «как в тот же день неделю назад» для той же кофейни (`groupby("shop_id")["revenue_k"].shift(7)` после сортировки по кофейне и дате);
- `mae_lin` — `LinearRegression` на признаках `X` (обучение на `tr`).

Оба — 2 знака. В `lin_better` — `True`, если линейная модель лучше «недели назад».

*Подсказки (2): `dsc.hint('baselines')`*

In [ ]:
%%task ch10/05-project/baselines
# уже есть: s (таблица) и prep(s) из шага 2
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

s = s.sort_values(["shop_id", "date"]).reset_index(drop=True)
X = prep(s)
y = s["revenue_k"]
tr = s["date"] < "2025-07-01"

### Шаг 4. Взаимодействия руками · база

Помогите линейной модели: добавьте к `X` три признака-произведения — `bc_off` (бизнес-центр × (выходной или праздник)), `mall_off` (ТЦ × (выходной или праздник)) и `street_rain` (улица × дождь). Обучите `LinearRegression` и сохраните `mae_inter` (2 знака) и `gain` — на сколько тысяч рублей ошибка меньше, чем у простой линейной модели (2 знака).

*Подсказки (2): `dsc.hint('interactions')`*

In [ ]:
%%task ch10/05-project/interactions
# уже есть: s, X, y, tr и mae_lin — ошибка простой линейной модели
off = ((X["weekend"] + X["holiday"]) > 0).astype(int)     # выходной или праздник

### Шаг 5. Бустинг и лес · база

На исходных признаках `X` (без ручных взаимодействий) обучите `HistGradientBoostingRegressor(random_state=0)` и `RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=0)`. Сохраните `mae_hgb`, `mae_rf` (2 знака) и `best_model` — `"бустинг"`, `"лес"` или `"линейная с взаимодействиями"` (её ошибка — `mae_inter` в заготовке).

*Подсказки (2): `dsc.hint('boosting')`*

In [ ]:
%%task ch10/05-project/boosting
# уже есть: s, X, y, tr и mae_inter — ошибка линейной модели с взаимодействиями
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

### Шаг 6. Подбор на валидации · посложнее

Подберите бустинг честно. Валидация — апрель–июнь 2025 (`val`: обучающие строки с датой от 2025-04-01), обучение для подбора — даты до 2025-04-01 (`fit`). Для каждой пары `learning_rate` в `[0.05, 0.1]` и `max_depth` в `[3, 6, None]` обучите `HistGradientBoostingRegressor(learning_rate=..., max_depth=..., max_iter=500, early_stopping=False, random_state=0)` на `fit`, по `staged_predict` найдите лучшее число итераций на `val` и его MAE. Сохраните DataFrame `grid` (столбцы `lr`, `depth`, `n_iter`, `val_mae`), лучшую строку — в `best` (Series), затем обучите модель с лучшими параметрами и `max_iter=n_iter` на **всём** обучении (`tr`) и сохраните `test_mae` (2 знака). Осторожно: в DataFrame глубина `None` превращается в `NaN`, а `HistGradientBoostingRegressor(max_depth=nan)` падает с ошибкой. Переведите её обратно: `None if pd.isna(best["depth"]) else int(best["depth"])`.

*Подсказки (2): `dsc.hint('tune')`*

In [ ]:
%%task ch10/05-project/tune
# уже есть: s, X, y, tr
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

fit = s["date"] < "2025-04-01"
val = tr & ~fit

### Шаг 7. Ловушка тренда · посложнее

Посмотрите на ошибки бустинга по умолчанию (`HistGradientBoostingRegressor(random_state=0)`) на тесте по месяцам: средняя ошибка `факт − прогноз` больше нуля значит, что модель занижает прогноз. Сеть растёт, а деревья не умеют продолжать тренд за пределы обучающих данных: для всех дат после июня 2025 `day_n` «как самый поздний известный день».

Исправим: оцените тренд линейной регрессией логарифма выручки на `day_n` по обучающим строкам, `trend = np.exp(модель.predict(s[["day_n"]]))`. Обучите бустинг (`random_state=0`) на признаках **без** `day_n` с ответом `y / trend`, а прогноз умножьте обратно на `trend`. Сохраните `bias_by_month` — среднюю ошибку `факт − прогноз` бустинга по умолчанию по месяцам теста (Series с индексом-номером месяца, 2 знака), `bias_before` и `bias_after` — среднюю ошибку `факт − прогноз` на всём тесте до и после (2 знака), `mae_after` (2 знака) и `growth` — годовой рост по тренду (`np.exp(коэффициент × 365)`, 3 знака).

*Подсказки (3): `dsc.hint('trend')`*

In [ ]:
%%task ch10/05-project/trend
# уже есть: s (со столбцом day_n), X, y, tr
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

### Шаг 8. Сравнение: качество и время · посложнее

Операционный директор спрашивает не только «насколько точно», но и «сколько это стоит в работе»: модель будут переобучать каждую ночь на свежих данных. Сведите модели проекта в одну таблицу. Для каждой модели из словаря `models` в заготовке обучите её на `tr`, спрогнозируйте тест и запишите:

- `rmse` и `mae` — ошибки на тесте, тыс. ₽ (2 знака);
- `fit_s` — время обучения в секундах, `predict_ms` — время прогноза на тесте в миллисекундах (`time.perf_counter()`).

Сохраните `table` (индекс — названия моделей, строки по возрастанию `rmse`), `best` — лучшую модель по `rmse`, `gain_vs_linear` — на сколько тыс. ₽ RMSE линейной модели больше, чем у лучшей (по значениям таблицы, 2 знака), и `nightly_ok` — `True`, если каждая модель обучается быстрее минуты.

Дальше — в своём Jupyter: в блокноте [ch10-catboost-lightgbm.ipynb](../notebooks/extra/ch10-catboost-lightgbm.ipynb) к той же таблице добавлены CatBoost и LightGBM; разбираем их в уроке «CatBoost и LightGBM в своём Jupyter» сразу после проекта. В браузере их нет, а в блокноте строка таблицы для них собирается так же:

```python static
# X_raw — те же признаки, но кофейня и формат строками (без get_dummies);
# fit — до апреля 2025, val — апрель–июнь: по нему ранняя остановка выбирает число деревьев
cb = CatBoostRegressor(iterations=3000, learning_rate=0.05, random_seed=0, verbose=0, early_stopping_rounds=100)
t0 = time.perf_counter()
cb.fit(X_raw[fit], y[fit], cat_features=["shop_id", "format"], eval_set=(X_raw[val], y[val]), use_best_model=True)
fit_s = time.perf_counter() - t0
```

*Подсказки (3): `dsc.hint('ch10/05-project/compare')`*

In [ ]:
%%task ch10/05-project/compare
# уже есть: s, X, y, tr
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

models = {
    "линейная": LinearRegression(),
    "лес": RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=0),
    "GradientBoosting": GradientBoostingRegressor(random_state=0),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
}

## Урок 8. CatBoost и LightGBM в своём Jupyter

В работе бустинг чаще берут из специализированных библиотек — CatBoost и LightGBM. Внутри у них тот же бустинг, что мы собирали руками, а снаружи — интерфейс как у sklearn. Запускаем обе библиотеки в своём Jupyter на данных проекта о выручке кофеен и сравниваем модели таблицей «качество и время».

Теория — на сайте курса: `ch10/06-libraries.html`.

### Таблица «качество и время» · база

Соберите в браузере такую же таблицу, как в блокноте с CatBoost и LightGBM, для трёх моделей sklearn на доставках. Модели и разбиение уже в заготовке. Для каждой модели замерьте `time.perf_counter()` время обучения в секундах (`fit_s`) и время прогноза на тесте в миллисекундах (`predict_ms`), посчитайте RMSE на тесте (`rmse`, 3 знака). Сохраните DataFrame `table`: индекс — названия моделей из словаря `models`, столбцы `rmse`, `fit_s`, `predict_ms`, строки отсортированы по `rmse` от лучшей. Ещё сохраните `best` — название модели с наименьшей ошибкой и `fastest_fit` — название модели, которая обучилась быстрее всех.

*Подсказки (3): `dsc.hint('ch10/06-libraries/speed-table')`*

In [ ]:
%%task ch10/06-libraries/speed-table
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=200, max_depth=3, random_state=0),
    "RandomForest": RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0),
}

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()